# Full-stroke radius experiment

Compare 5, 7, and 9 mm radii by **takeoff kinetic energy**, sweeping total body mass from 20 to 500 g. The motor stays powered until the mechanical stroke ends. There is no pulse cutoff.

- Total length: 180 mm; dead length: 56.571 mm; usable stroke: 123.429 mm.
- Dead/total = 49.5/157.5. Foot mass is zero. Start from rest and zero twist.
- Primary case retains the notebook's measured rotor inertia, 1.86e-6 kg m².
- A second case explores a different motor/drivetrain inertia, 1.82396e-5 kg m². Treat this as a separate hardware design, with its rotating mass included when checking physical feasibility.
- The plots use simulated linear torque–speed curves. The 5 V value is the reference for the envelope check, not the nonlinear experimental 5 V response. Motor curves are linear in speed. They are design targets inside the measured torque–speed envelope, not a claim that an unmodified voltage command produces those exact curves.

Angle-domain kinetic-energy integration covers the complete stroke. With q = dy/dθ and total kinetic energy K, ω = sqrt(2K/(I + m q²)) and dK/dθ = τ(ω) − mgq. At takeoff, the body's share is K × m q²/(I + m q²).


The bottom row overlays each used linear motor curve on the fitted experimental envelope across 1–10 V, with the 5 V bound shown as a dashed reference. The envelope is the same conservative bound used by the motor checks.


The linear motor parameters are reoptimized separately for each fixed inertia at 180 mm. The objective is the minimum of the four adjacent-radius energy advantages at 20 g and 500 g, with all predicted jumps above 0.15 m. The search height floor is 0.15005 m. The plotted settings are rounded to 0.1 mN m and 1 rpm and independently verified. Maximum shaft power is marked on each motor curve; for a linear curve it is stall torque × free angular speed / 4.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

length = 0.180
ratio = 49.5 / 157.5
stroke = length * (1 - ratio)
g = 9.81
voltage = 5.0
radii = [0.005, 0.007, 0.009]
masses = np.linspace(0.020, 0.500, 161)
cases = {
    'Current rotor': {'inertia': 1.86e-6, 'stall': 0.1164, 'free_rpm': 5751.0},
    'Higher inertia': {'inertia': 1.82396e-5, 'stall': 0.1090, 'free_rpm': 4445.0},
}

# Conservative bound from the measured hard-knee and smooth voltage fits.
free_data = np.array([2049.7002, 4224.7279, 6390.3895, 8368.1081, 10005.7876,
                      12022.9659, 14866.4361, 17137.8066, 19872.9528, 21643.4236])

def envelope(volts, rpm):
    free_rpm = np.interp(volts, np.arange(1, 11), free_data)
    hard = 4.70391607e-5 - 4.52823176e-6 * min(volts, 5.352084)
    soft_v = volts - 0.683395924 * np.logaddexp(0, (volts - 4.84445085) / 0.683395924)
    smooth = 4.84787835e-5 - 5.32297125e-6 * soft_v
    return np.minimum(min(hard, smooth) * (free_rpm - rpm), 0.25999)

def energy(mass, radius, case):
    inertia = case['inertia']
    free = case['free_rpm'] * 2 * np.pi / 60
    end = length * np.sqrt(1 - ratio**2) / radius

    def dynamics(theta, state):
        root = np.sqrt(length**2 - (radius * theta)**2)
        q = radius**2 * theta / root
        omega = np.sqrt(2 * max(state[0], 0) / (inertia + mass * q**2))
        torque = case['stall'] * (1 - omega / free)
        return [torque - mass * g * q]

    sol = solve_ivp(dynamics, (0, end), [0.0], method='DOP853',
                    rtol=1e-9, atol=1e-11)
    assert sol.success and sol.y[0, -1] > 0, 'Stroke not completed'
    q = radius * np.sqrt(1 - ratio**2) / ratio
    return sol.y[0, -1] * mass * q**2 / (inertia + mass * q**2)


In [2]:
rows = []
for name, case in cases.items():
    for mass in masses:
        for radius in radii:
            ke = energy(mass, radius, case)
            rows.append({'case': name, 'mass_g': mass * 1000, 'radius_mm': radius * 1000,
                         'energy_J': ke, 'height_m': ke / (mass * g)})
results = pd.DataFrame(rows)
assert results['height_m'].min() > 0.15
print(f"Minimum jump height: {100 * results['height_m'].min():.1f} cm (target: 15 cm)")

summary = []
crossovers = []
for name, case in cases.items():
    for mass in [masses[0], masses[-1]]:
        e = np.array([energy(mass, r, case) for r in radii])
        h = e / (mass * g)
        sorted_e = np.sort(e)
        summary.append({'case': name, 'mass_g': mass * 1000,
                        'r5_height_m': h[0], 'r7_height_m': h[1], 'r9_height_m': h[2],
                        'winner_mm': radii[e.argmax()] * 1000,
                        'winner_lead_pct': 100 * (sorted_e[-1] / sorted_e[-2] - 1)})
    for a, b in [(0.009, 0.007), (0.007, 0.005)]:
        mass = brentq(lambda m: energy(m, a, case) - energy(m, b, case), 0.020, 0.500)
        crossovers.append({'case': name, 'change': f'{a*1000:.0f} → {b*1000:.0f} mm',
                           'mass_g': mass * 1000})
summary = pd.DataFrame(summary)
crossovers = pd.DataFrame(crossovers)
print(summary.round(4).to_string(index=False))
print('Crossovers:')
print(crossovers.round(2).to_string(index=False))

# Energy and separation use the same linear mass axis in both designs.
colors = ['#563091', '#d36a13', '#148b82']
speed = np.linspace(0, free_data[-1], 601)
limit = np.maximum.reduce([envelope(v, speed) for v in range(1, 11)])
reference_speed = np.linspace(0, np.interp(voltage, np.arange(1, 11), free_data), 301)
fig, axes = plt.subplots(3, 2, figsize=(12, 11), sharex='row',
                         sharey='row', height_ratios=[1.25, 1, 1])
for col, (name, case) in enumerate(cases.items()):
    data = results[results['case'] == name].pivot(
        index='mass_g', columns='radius_mm', values='energy_J')
    crossings = crossovers[crossovers['case'] == name]['mass_g'].to_numpy()
    for mass in crossings:
        data.loc[mass] = [energy(mass / 1000, r, case)
                          for r in radii]
    data = data.sort_index()
    x = data.index.to_numpy()
    energies = data.to_numpy()
    ordered = np.sort(energies, axis=1)
    lead = 100 * (ordered[:, -1] / ordered[:, -2] - 1)
    top, bottom, motor = axes[:, col]
    for i, (radius, color) in enumerate(zip(radii, colors)):
        top.plot(x, energies[:, i], color=color, lw=2.3, label=f'{radius * 1000:.0f} mm')
        top.scatter(x[[0, -1]], energies[[0, -1], i], color=color, s=22, zorder=4)
    top.set_title(f"{name} (I = {case['inertia']:.6g} kg m²)\n"
                  f"Stall torque {case['stall'] * 1000:g} mN m; free speed {case['free_rpm']:g} rpm",
                  fontsize=11)
    top.legend(title='Radius', loc='lower left', ncol=3, frameon=False)
    for i, color in enumerate(colors):
        top.annotate(f'{energies[-1, i]:.3f} J', (500, energies[-1, i]),
                     xytext=(-5, 7), textcoords='offset points',
                     ha='right', color=color, fontsize=9)
    for mass in crossings:
        top.axvline(mass, color='0.65', lw=0.9, ls=':')
    edges = np.r_[20, crossings, 500]
    for left, right, i in zip(edges[:-1], edges[1:], [2, 1, 0]):
        mask = (x >= left) & (x <= right)
        bottom.plot(x[mask], lead[mask], color=colors[i], lw=2.6)
        bottom.axvspan(left, right, color=colors[i], alpha=0.06, lw=0)
        bottom.text((left + right) / 2, 1.03, f'{radii[i] * 1000:.0f} mm\nbest',
                    transform=bottom.get_xaxis_transform(),
                    ha='center', va='bottom', color=colors[i], fontsize=10)
    for mass in crossings:
        bottom.axvline(mass, color='0.55', lw=0.9, ls=':')
        bottom.text(mass + 5, 17, f'{mass:.0f} g', rotation=90,
                    va='center', fontsize=9, color='0.4')
    bottom.axhline(10, color='0.35', lw=1, ls='--')
    bottom.text(240, 10.7, '10% target', color='0.35', fontsize=9)
    for mass, margin, color, align in zip(x[[0, -1]], lead[[0, -1]],
                                          [colors[2], colors[0]], ['left', 'right']):
        bottom.scatter(mass, margin, color=color, s=24, zorder=4)
        bottom.text(mass, margin + 0.8, f'{margin:.1f}%', color=color,
                    ha=align, va='bottom', fontsize=10)
    bottom.set(xlabel='Total moving body mass (g)', ylim=(0, 25))
    top.tick_params(labelbottom=False)
    motor.fill_between(speed, 0, 1000 * limit, color='0.88',
                       label='Experimental envelope (1–10 V)')
    motor.plot(speed, 1000 * limit, color='0.55', lw=1)
    motor.plot(reference_speed, 1000 * envelope(voltage, reference_speed),
               color='#2875a7', ls='--', lw=1.7, label=f'Experimental {voltage:g} V reference')
    model_speed = np.linspace(0, case['free_rpm'], 201)
    model_torque = case['stall'] * (1 - model_speed / case['free_rpm'])
    motor.plot(model_speed, 1000 * model_torque, color='black', lw=2.5,
               label='Used linear model')
    for rpm, torque, color, offset in [
        (speed, limit, '0.35', (-75, -26)),
        (reference_speed, envelope(voltage, reference_speed), '#2875a7', (12, 12)),
        (model_speed, model_torque, 'black', (12, 12)),
    ]:
        power = torque * rpm * 2 * np.pi / 60
        peak = power.argmax()
        point = (rpm[peak], 1000 * torque[peak])
        motor.scatter(*point, color=color, s=22, zorder=4)
        motor.annotate(f'{power[peak]:.1f} W max', point, xytext=offset,
                       textcoords='offset points', color=color, fontsize=8,
                       arrowprops={'arrowstyle': '-', 'color': color, 'lw': 0.7})
    motor.set(title='Motor curves; dots mark maximum shaft power',
              xlabel='Motor speed (rpm)', xlim=(0, 22000), ylim=(0, 280),
              xticks=[0, 5000, 10000, 15000, 20000])
    motor.title.set_fontsize(10)
    motor.legend(loc='upper right', fontsize=8, frameon=False)
    for ax in axes[:2, col]:
        ax.set(xlim=(20, 500), xticks=[20, 100, 200, 300, 400, 500])
    for ax in axes[:, col]:
        ax.grid(axis='y', alpha=0.18)
        ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].set(ylabel='Takeoff kinetic energy (J)', ylim=(0, 1.28))
axes[1, 0].set_ylabel("Winner's energy advantage\nover next best radius (%)")
axes[2, 0].set_ylabel('Motor torque (mN m)')
fig.suptitle('Takeoff energy vs body mass', fontsize=16, y=0.99)
fig.text(0.5, 0.943, f'Linear motor model; {length * 1000:g} mm total length; '
         f'{length * ratio * 1000:.1f} mm dead length; zero foot mass; full stroke',
         ha='center', fontsize=10)
fig.text(0.5, 0.014, 'Ideal-model predictions. Higher inertia requires a hardware change. '
         f"Minimum jump height: {100 * results['height_m'].min():.1f} cm (target: 15 cm).", ha='center', fontsize=9, color='0.35')
fig.tight_layout(rect=(0, 0.04, 1, 0.925), h_pad=1.8)
figures = [fig]
fig.savefig('results/full_stroke_radius_final.png', dpi=220, bbox_inches='tight')


Minimum jump height: 15.0 cm (target: 15 cm)
          case  mass_g  r5_height_m  r7_height_m  r9_height_m  winner_mm  winner_lead_pct
 Current rotor    20.0       1.8355       2.3197       2.5762        9.0          11.0576
 Current rotor   500.0       0.1949       0.1755       0.1501        5.0          11.0546
Higher inertia    20.0       1.4005       1.9535       2.3122        9.0          18.3620
Higher inertia   500.0       0.2180       0.1842       0.1501        5.0          18.3515
Crossovers:
          case   change  mass_g
 Current rotor 9 → 7 mm   99.67
 Current rotor 7 → 5 mm  228.97
Higher inertia 9 → 7 mm   82.62
Higher inertia 7 → 5 mm  172.06


In [3]:
# Check both target motor families against the conservative measured envelope.
for case in cases.values():
    for v in np.linspace(1, 10, 91):
        speed = np.linspace(0, case['free_rpm'] * v / voltage, 201)
        available = envelope(v, speed)
        target = case['stall'] * v / voltage * (1 - speed / (case['free_rpm'] * v / voltage))
        assert np.all(target <= available + 1e-10)

# Independently verify selected points by integrating angle and speed in time.
for name, case in cases.items():
    for mass in [0.020, 0.100, 0.250, 0.500]:
        for radius in radii:
            end = length * np.sqrt(1 - ratio**2) / radius
            free = case['free_rpm'] * 2 * np.pi / 60
            inertia = case['inertia']
            def dynamics(t, state):
                theta, omega = state
                root = np.sqrt(length**2 - (radius * theta)**2)
                q = radius**2 * theta / root
                dq = radius**2 * length**2 / root**3
                torque = case['stall'] * (1 - omega / free)
                return [omega, (torque - mass*g*q - mass*q*dq*omega**2) / (inertia + mass*q**2)]
            def stop(t, state):
                return state[0] - end
            stop.terminal = True
            sol = solve_ivp(dynamics, (0, 2), [0, 0], events=stop, method='DOP853',
                            rtol=1e-10, atol=1e-12, max_step=0.001)
            assert sol.success and len(sol.t_events[0]) == 1
            omega = sol.y_events[0][0, 1]
            q = radius * np.sqrt(1 - ratio**2) / ratio
            np.testing.assert_allclose(energy(mass, radius, case), 0.5*mass*(q*omega)**2, rtol=1e-6)
print('Verified complete-stroke predictions and motor-envelope constraints.')


Verified complete-stroke predictions and motor-envelope constraints.


## How to run the experiment

Keep voltage, motor curve, string length, dead/total ratio, inertia, and total moving mass identical across the three radii. Drive the motor through the entire stroke. Measure takeoff velocity and calculate K = ½mv²; h = v²/(2g) is the rise above the takeoff position. Measuring velocity avoids needing to film the full flight.

Use endpoint masses to demonstrate the clearest reversal, then add masses around the reported crossovers. Five repeated trials per radius and mass, in alternating radius order, give a useful first check of repeatability. Keep initial twist, string condition, and motor temperature consistent.

The 10% criterion applies to separation at selected operating points away from a crossover. The curves must tie at a crossover. The 7 mm radius has a narrow winning region, with a smaller advantage than 10%; this search did not establish a full-stroke design where every radius wins by 10% while every compared jump exceeds 0.15 m.

At 180 mm, the current-rotor model is reoptimized to 116.4 mN m stall torque and 5751 rpm free speed, giving 17.5 W maximum shaft power. Its endpoint winner advantages are 11.1% at both 20 g and 500 g, and its crossovers are 99.7 g and 229.0 g. Predicted rises span 0.1501–2.58 m.

With inertia 1.82396e-5 kg m², the reoptimized model uses 109.0 mN m stall torque and 4445 rpm free speed, giving 12.7 W maximum shaft power. Endpoint winner advantages are 18.4% at both 20 g and 500 g, with crossovers at 82.6 g and 172.1 g. Predicted rises span 0.1501–2.31 m. This case requires a separate inertia and rotating-mass feasibility check.

Both searches maximize the smallest adjacent-radius energy gap at the endpoint masses, rather than maximizing motor power. Three differential-evolution seeds agree, followed by refinement with the adaptive full-stroke solver. Both rounded models satisfy the experimental envelope checks and the 0.15 m height floor across the complete plotted mass sweep.

These are ideal-model predictions. They omit transmission losses and compliance. Bench-check the chosen linear torque–speed curve and actual moving mass before using these settings as an experimental claim.


In [4]:
results.to_csv('results/mass_sweep.csv', index=False)
summary.to_csv('results/endpoints.csv', index=False)
crossovers.to_csv('results/crossovers.csv', index=False)
